# Kappa synchrony measurements for biohphysical EI network model and convolved single unit spiking data

In [ ]:
%pip install jaxley

import os
# os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=16'

from jax import config
config.update("jax_enable_x64", True)
config.update("jax_platform_name", "cpu") # 'gpu' / 'cpu'
# config.update("jax_debug_nans", True)

import jax
import numpy as np
import jaxley as jx
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxley.optimize.transforms as jt

from jax import jit, vmap, value_and_grad
from jaxley.channels import Leak, HH
from jaxley.synapses import IonotropicSynapse
from jaxley.connect import fully_connect, sparse_connect, connect
from scipy import ndimage
import scipy
import pickle


# figure_save_path = "/Computational/GSDR/Figures"
# model_save_path = "/Computational/GSDR/NeuralCircuits"
# save_dir = "/Computational/GSDR/SignalArrays/"

# figure_save_path = "/content/data/Figures"
# model_save_path = "/content/data/NeuralCircuits"
# save_dir = "/content/data/SignalArrays/"

from google.colab import drive
drive.mount('/content/drive/')
figure_save_path = "/content/drive/MyDrive/Workspace/Computational/Figures/" # Drive
model_save_path = "/content/drive/MyDrive/Workspace/Computational/Models/" # Drive
save_dir = "/content/drive/MyDrive/Workspace/Analysis/npy/" # Drive

file_name = "oxm0818_units.npy"

os.makedirs(figure_save_path, exist_ok=True)
os.makedirs(model_save_path, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, file_name)
selected_unit_spiking = jnp.load(save_path)
print(f"'selected_unit_spiking' loaded from {save_path}")
print(f"Shape of loaded_selected_unit_spiking: {selected_unit_spiking.shape}")

### Power-spectrum density in trials


Get single neuron spiking array : trial/unit/sample

Sub-select units of a given area based on info : trial/selectedunit/sample

Calculate individial psds : trial/selectedunit/psd_sample

Average into population psd : trial/psd_sample

Plot1 : image (x-axis frequency, y-axis trials)

0818: PFC; TEO, FST; MT, MST

0825: PFC; MT, MST; V4, TEO


In [ ]:
file_name_1 = "oxm0825_units.npy"
file_name_2 = "oxm0825_units_info.npy"
all_unit_spiking_1 = np.load(save_dir + file_name_1)
all_unit_info_1 = np.load(save_dir + file_name_2, allow_pickle=True)

# file_name_1 = "oxm0818_units.npy"
# file_name_2 = "oxm0818_units_info.npy"

# all_unit_spiking_2 = np.load(save_dir + file_name_1)
# all_unit_info_2 = np.load(save_dir + file_name_2, allow_pickle=True)

In [ ]:
idx = np.zeros([2, 3], np.int32)
for ik in range(3):
    idx[0, ik] = all_unit_info_1[ik]['ids'].shape[0]
    # idx[1, ik] = all_unit_info_2[ik]['ids'].shape[0]

for ik in range(2):
    idx[0, ik+1] = idx[0, ik+1] + idx[0, ik]
    # idx[1, ik+1] = idx[1, ik+1] + idx[1, ik]

In [ ]:
# pfc_1_units = all_unit_spiking_2[:, 0:idx[0, 0], :2000]
# teofst_1_units = all_unit_spiking_2[:, idx[0, 0]:idx[0, 1], :2000]
# mtmst_1_units = all_unit_spiking_2[:, idx[0, 1]:idx[0, 2], :2000]

pfc_2_units = all_unit_spiking_1[:, 0:idx[0, 0], :]
mtmst_2_units = all_unit_spiking_1[:, idx[0, 0]:idx[0, 1], :2000]
v4teo_2_units = all_unit_spiking_1[:, idx[0, 1]:idx[0, 2], :2000]

In [ ]:
# selected_unit_spiking_1 = mtmst_1_units
selected_unit_spiking_2 = mtmst_2_units
# selected_unit_spiking_3 = pfc_1_units
selected_unit_spiking_4 = pfc_2_units

# selected_unit_spiking_1 = np.reshape(selected_unit_spiking_1, [selected_unit_spiking_1.shape[0], selected_unit_spiking_1.shape[2], -1])
selected_unit_spiking_2 = np.reshape(selected_unit_spiking_2, [selected_unit_spiking_2.shape[0], selected_unit_spiking_2.shape[2], -1])
# selected_unit_spiking_3 = np.reshape(selected_unit_spiking_3, [selected_unit_spiking_3.shape[0], selected_unit_spiking_3.shape[2], -1])
selected_unit_spiking_4 = np.reshape(selected_unit_spiking_4, [selected_unit_spiking_4.shape[0], selected_unit_spiking_4.shape[2], -1])

# print(selected_unit_spiking_1.shape)
print(selected_unit_spiking_2.shape)
# print(selected_unit_spiking_3.shape)
print(selected_unit_spiking_4.shape)

In [ ]:
def get_signal_psd(traces, fs=1000):

    signal = jnp.nanmean(jnp.squeeze(traces), axis=1) # Average voltage across all recorded neurons over time
    N = signal.shape[-1]

    signal_fft = jnp.fft.rfft(signal)
    freqs = jnp.fft.rfftfreq(N, 1/fs)
    psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

    target_freqs = global_psd_interval
    interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)
    interpolated_psd = ndimage.uniform_filter1d(interpolated_psd, size=jnp.ceil(3*jnp.sqrt(fs/1000)).astype(int))

    interpolated_psd = interpolated_psd - jnp.min(interpolated_psd)
    max_psd = jnp.max(interpolated_psd)
    scaled_psd = interpolated_psd / (max_psd + 1e-6)
    return scaled_psd


def trial_psd_group(selected_unit_spikings, n_rows, n_fbins, k_rows):

    t_p1 = 1100
    t_p2 = 1700
    print(selected_unit_spikings.shape)

    labels_g = jnp.zeros((n_rows, n_fbins))
    for ik in range(n_rows):

        temp_signal = selected_unit_spikings[ik*k_rows:ik*k_rows+k_rows, t_p1:t_p2, :]
        temp_signal = scipy.ndimage.median_filter(temp_signal, size=(1, 3, 1)) # apply median filter on temp signal
        stim_units_spectrum_g = get_signal_psd(jnp.nanmean(temp_signal, 0))
        labels_g = labels_g.at[ik].set(stim_units_spectrum_g)

    return labels_g


def get_hsnr_units(unit_spiking_array, percent_thresh=0.5):
    """
    Filters units based on their average activity falling within a middle percentile interval.

    Args:
        unit_spiking_array (jnp.ndarray): Input array with shape (trials, samples, neuron_id).
        percent_thresh (float): The middle interval percentage (e.g., 0.5 for 50%).
                                Units in the (1-percent_thresh)/2 to (1+(1-percent_thresh)/2) percentile
                                of average firing rate will be selected.

    Returns:
        jnp.ndarray: A new array with shape (trials, samples, n_selected_units),
                     containing only the selected units.
    """
    if not (0 <= percent_thresh <= 1):
        raise ValueError("percent_thresh must be between 0 and 1.")

    # Calculate average activity (proxy for firing rate) for each neuron
    # across all trials and samples.
    avg_activity_per_neuron = jnp.mean(unit_spiking_array, axis=(0, 1))

    # Get the sorted indices of the average activities
    sorted_indices = jnp.argsort(avg_activity_per_neuron)

    num_neurons = unit_spiking_array.shape[2]

    # Calculate the lower and upper bounds for the percentile interval
    lower_bound_idx = int(num_neurons * (1 - percent_thresh) / 2)
    upper_bound_idx = int(num_neurons * (1 + percent_thresh) / 2)

    # Ensure indices are within bounds and upper_bound_idx is at least lower_bound_idx
    lower_bound_idx = max(0, lower_bound_idx)
    upper_bound_idx = min(num_neurons, upper_bound_idx)
    if upper_bound_idx <= lower_bound_idx and num_neurons > 0:
        # If the range is too small, select at least one unit or adjust bounds
        if num_neurons == 1: # If only one neuron, select it
            selected_unit_indices = sorted_indices
        else:
            # Try to select at least a small central chunk if possible
            mid_point = num_neurons // 2
            selected_unit_indices = sorted_indices[max(0, mid_point - 1) : min(num_neurons, mid_point + 1)]
    else:
        selected_unit_indices = sorted_indices[lower_bound_idx:upper_bound_idx]

    # Select the units from the original array using the determined indices
    hsnr_units = unit_spiking_array[:, :, selected_unit_indices]

    return hsnr_units

## MTMST1

In [ ]:
global_psd_interval = jnp.linspace(2.0, 100.0, 200)

print(selected_unit_spiking_2.shape)
selected_unit_spiking = get_hsnr_units(selected_unit_spiking_2, 0.8)
print(selected_unit_spiking.shape)

k_rows = 2
n_fbins = global_psd_interval.shape[0]
n_rows = int(selected_unit_spiking.shape[0]/k_rows)
labels_g = trial_psd_group(selected_unit_spiking, n_rows, n_fbins, k_rows)

plt.figure(figsize=(10, 6))

plt.imshow(labels_g, aspect='auto', origin='lower',
           extent=[global_psd_interval[0], global_psd_interval[-1], 0, n_rows])

plt.title("Target Power Spectral Densities (MT/MST 230825)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Trial Group Index")
plt.colorbar(label='Normalized Power')
plt.tight_layout()
plt.show()

In [ ]:
global_psd_interval = jnp.linspace(30.0, 100.0, 200)

print(selected_unit_spiking_1.shape)
selected_unit_spiking = get_hsnr_units(selected_unit_spiking_1, 0.2)
print(selected_unit_spiking.shape)

k_rows = 2
n_fbins = global_psd_interval.shape[0]
n_rows = int(selected_unit_spiking.shape[0]/k_rows)
labels_g = trial_psd_group(selected_unit_spiking, n_rows, n_fbins, k_rows)

plt.figure(figsize=(10, 6))

plt.imshow(labels_g, aspect='auto', origin='lower',
           extent=[global_psd_interval[0], global_psd_interval[-1], 0, n_rows])

plt.title("Power Spectral Densities (MT/MST, ses230818)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Trial Group Index")
plt.colorbar(label='Normalized Power')
plt.tight_layout()
plt.show()

In [ ]:
global_psd_interval = jnp.linspace(2.0, 100.0, 200)

selected_unit_spiking = get_hsnr_units(selected_unit_spiking_3, 0.8)
print(selected_unit_spiking.shape)

k_rows = 4
n_fbins = global_psd_interval.shape[0]
n_rows = int(selected_unit_spiking.shape[0]/k_rows)
labels_g = trial_psd_group(selected_unit_spiking, n_rows, n_fbins, k_rows)

plt.figure(figsize=(10, 6))

plt.imshow(labels_g, aspect='auto', origin='lower',
           extent=[global_psd_interval[0], global_psd_interval[-1], 0, n_rows])

plt.title("Power Spectral Densities (PFC, ses230818)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Trial Group Index")
plt.colorbar(label='Normalized Power')
plt.tight_layout()
plt.show()

In [ ]:
# imx = np.mean(selected_unit_spiking_1, 2)
# plt.figure(figsize=(10, 6))

# plt.imshow(imx.T, aspect='auto', origin='lower')

# plt.title("mean fr-t")
# plt.xlabel("time ms")
# plt.ylabel("u")
# plt.colorbar(label='au')
# plt.tight_layout()
# plt.show()